In [16]:
import numpy as np
import pandas as pd
import h5py
import os
from gemini_util_light import PreloadedEventGenerator

In [3]:
def load_events(data_paths, event_metadata_path='./event_metadata.csv', limit=None, parts=None, shuffle_train_dev=False, custom_split=None, data_keys=None,
                overwrite_sampling_rate=None, min_mag=None, mag_key=None, decimate_events=None):
    if min_mag is not None and mag_key is None:
        raise ValueError('mag_key needs to be set to enforce magnitude threshold')
    if isinstance(data_paths, str):
        data_paths = [data_paths]
    if len(data_paths) > 1:
        raise NotImplementedError('Loading partitioned data is currently not supported')
    data_path = data_paths[0]

    if not os.path.exists(event_metadata_path):
        build_event_metadata(data_path, event_metadata_path, overwrite_sampling_rate)
    event_metadata = pd.read_csv(event_metadata_path)
    if min_mag is not None:
        event_metadata = event_metadata[event_metadata[mag_key] >= min_mag]
    for event_key in ['KiK_File', '#EventID', 'EVENT']:
        if event_key in event_metadata.columns:
            break

    if limit:
        event_metadata = event_metadata.iloc[:limit]
    if parts:
        mask = TrainDevTestSplitter.run_method(event_metadata, custom_split, shuffle_train_dev, parts=parts)
        event_metadata = event_metadata[mask]

    if decimate_events is not None:
        event_metadata = event_metadata.iloc[::decimate_events]

    metadata = {}
    data = {}

    with h5py.File(data_path, 'r') as f:
        for key in f['metadata'].keys():
            if key == 'event_metadata':
                continue
            #metadata[key] = f['metadata'][key].value
            metadata[key] = f['metadata'][key][()]

        if overwrite_sampling_rate is not None:
            if metadata['sampling_rate'] % overwrite_sampling_rate != 0:
                raise ValueError(f'Overwrite sampling ({overwrite_sampling_rate}) rate must be true divisor of sampling'
                                 f' rate ({metadata["sampling_rate"]})')
            decimate = metadata['sampling_rate'] // overwrite_sampling_rate
            metadata['sampling_rate'] = overwrite_sampling_rate
        else:
            decimate = 1

        skipped = 0
        contained = []
        n_rec_per_event = []
        for _, event in event_metadata.iterrows():
            event_name = str(event[event_key])
            if event_name not in f['data']:
                skipped += 1
                contained += [False]
                continue
            contained += [True]
            g_event = f['data'][event_name]
            for key in g_event:
                if key == 'waveforms':
                    cur_waveform = g_event[key][:, ::decimate, :]
                    n_rec_per_event.append(cur_waveform.shape[0])

        if len(contained) < len(event_metadata):
            contained += [True for _ in range(len(event_metadata) - len(contained))]
        event_metadata = event_metadata[contained]
        if skipped > 0:
            print(f'Skipped {skipped} events')

    return event_metadata, data, metadata

In [4]:
data_path = "../TEAM/data/italy/italy.hdf5"

In [5]:
metadata = {}
data = {}

In [6]:
f = h5py.File(data_path, 'r')

In [7]:
f['metadata'].keys()

<KeysViewHDF5 ['channels', 'event_metadata', 'highpass_pga', 'pga_thresholds', 'sampling_rate', 'time_after', 'time_before']>

In [8]:
for key in f['metadata'].keys():
    if key == 'event_metadata':
        continue
    #metadata[key] = f['metadata'][key].value
    metadata[key] = f['metadata'][key][()]

In [9]:
metadata

{'channels': array([b'N', b'E', b'Z'], dtype='|S4'),
 'highpass_pga': np.float64(0.2),
 'pga_thresholds': array([0.01, 0.02, 0.05, 0.1 , 0.2 ]),
 'sampling_rate': np.int64(100),
 'time_after': np.int64(25),
 'time_before': np.int64(5)}

In [10]:
decimate = 1

In [11]:
event_metadata = pd.read_hdf(data_path, 'metadata/event_metadata')

In [12]:
for event_key in ['KiK_File', '#EventID', 'EVENT']:
    if event_key in event_metadata.columns:
        break


In [13]:
skipped = 0
contained = []
n_rec_per_event = []
count = 0
for _, event in event_metadata.iterrows():
    event_name = str(event[event_key])
    if event_name not in f['data']:
        skipped += 1
        contained += [False]
        continue
    contained += [True]
    g_event = f['data'][event_name]
    break
    for key in g_event:
        if key == 'waveforms':
            cur_waveform = g_event[key][:, ::decimate, :]
            n_rec_per_event.append(cur_waveform.shape[0])
    calc_pp = np.linalg.norm(cur_waveform,axis=2)
    calc_pp =np.log10(np.max(np.abs(calc_pp),axis=1))
    print('pga: ', g_event['pga'][:], 'pgv: ',g_event['pgv'][:], 'calc_pp: ', calc_pp)
    count += 1
    if count >= 20:
        break

In [14]:
for key in g_event:
    print(key, g_event[key], g_event[key][:])

coords <HDF5 dataset "coords": shape (1, 3), type "<f8"> [[43.986    7.55317 -0.595  ]]
p_picks <HDF5 dataset "p_picks": shape (1,), type "<i8"> [500]
pga <HDF5 dataset "pga": shape (1,), type "<f8"> [-4.29437884]
pga_times <HDF5 dataset "pga_times": shape (1, 5), type "<i8"> [[0 0 0 0 0]]
pgv <HDF5 dataset "pgv": shape (1,), type "<f8"> [-5.38376071]
stations <HDF5 dataset "stations": shape (1,), type "|S10"> [b'FR.SAOF']
waveforms <HDF5 dataset "waveforms": shape (1, 3000, 3), type "<f8"> [[[ 2.20222468e-06  3.84043572e-06  4.02344075e-06]
  [ 1.66983531e-06  3.91607259e-06  7.75466567e-07]
  [ 1.45466253e-06  1.94601086e-06 -2.93929665e-07]
  ...
  [ 1.19136630e-05 -6.71024048e-06 -1.27644347e-05]
  [ 1.92368270e-05 -1.16906120e-06 -1.30430134e-05]
  [ 2.74979571e-05  2.45189743e-06 -1.67831942e-05]]]


In [15]:
f.close()

In [45]:
wv_norm = np.linalg.norm(cur_waveform,axis=2)

In [48]:
np.max(wv_norm,axis=1)

array([0.00036099])

In [43]:
np.log10(np.abs(wv_norm).max())

np.float64(-4.3955361755740725)

In [24]:
g_event.get('pga')

<HDF5 dataset "pga": shape (1,), type "<f8">

In [ ]:
with h5py.File(data_path, 'r') as f:
    for key in f['metadata'].keys():
        if key == 'event_metadata':
            continue
        #metadata[key] = f['metadata'][key].value
        metadata[key] = f['metadata'][key][()]

    if overwrite_sampling_rate is not None:
        if metadata['sampling_rate'] % overwrite_sampling_rate != 0:
            raise ValueError(f'Overwrite sampling ({overwrite_sampling_rate}) rate must be true divisor of sampling'
                             f' rate ({metadata["sampling_rate"]})')
        decimate = metadata['sampling_rate'] // overwrite_sampling_rate
        metadata['sampling_rate'] = overwrite_sampling_rate
    else:
        decimate = 1

    skipped = 0
    contained = []
    n_rec_per_event = []
    for _, event in event_metadata.iterrows():
        event_name = str(event[event_key])
        if event_name not in f['data']:
            skipped += 1
            contained += [False]
            continue
        contained += [True]
        g_event = f['data'][event_name]
        for key in g_event:
            if key == 'waveforms':
                cur_waveform = g_event[key][:, ::decimate, :]
                n_rec_per_event.append(cur_waveform.shape[0])

    if len(contained) < len(event_metadata):
        contained += [True for _ in range(len(event_metadata) - len(contained))]
    event_metadata = event_metadata[contained]
    if skipped > 0:
        print(f'Skipped {skipped} events')

In [ ]:
dataset = Pre